In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/SEPH_TIME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPI12_DECLINED_COMPLETE',
 'sepsis_cases_1',
 'sepsis_cases_4',
 'BPIC15_common']

In [4]:
dataset = "sepsis_cases_1"

In [5]:
if dataset.startswith("BPIC15"):
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,DiagnosticUrinarySediment,DiagnosticXthorax,DisfuncOrg,Hypotensie,Hypoxie,InfectionSuspected,Infusion,Oligurie,SIRSCritHeartRate,SIRSCritLeucos,SIRSCritTachypnea,SIRSCritTemperature,SIRSCriteria2OrMore,Age,CaseID,Activity,org:group,CRP,LacticAcid,Leucocytes,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases,label,remaining_time
0,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Registration,A,0.0,0.0,0.0,1.413955e+09,555,10,2,9,0.000000,0.000000,1,81,regular,968359.0
1,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,Leucocytes,B,0.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,2,81,regular,967680.0
2,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,CRP,B,21.0,0.0,9.6,1.413956e+09,567,10,2,9,0.000000,11.316667,3,81,regular,967680.0
3,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,LacticAcid,B,21.0,2.2,9.6,1.413956e+09,567,10,2,9,11.316667,11.316667,4,81,regular,967680.0
4,A,True,True,True,True,True,False,False,False,True,True,True,True,True,False,True,True,False,True,False,True,True,True,85.0,A,ER Triage,C,21.0,2.2,9.6,1.413956e+09,573,10,2,9,6.616667,17.933333,5,81,regular,967283.0


In [8]:
import pandas as pd

tab_test = pd.read_csv(data_dir_processed + f"{dataset}_processed_test.csv")

# time:timestamp should be in float representing number of seconds.
durations = (
    tab_test
    .groupby("CaseID")["time:timestamp"]
    .agg(start="min", end="max")
)
durations["duration_sec"] = (durations["end"] - durations["start"])

# 4) Compute averages
avg_sec  = durations["duration_sec"].mean()
avg_days = avg_sec / (24 * 3600)

print(f"Average trace duration (test set): {avg_sec:,.0f} seconds\n")
print(f"which means ≃ {avg_days:.1f} days")


Average trace duration (test set): 651,358 seconds

which means ≃ 7.5 days


In [9]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [10]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

In [11]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures, Compose

#transform = ToUndirected()
transform = Compose([ToUndirected(), NormalizeFeatures()])

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for i in range(len(X_test)):
                X_test[i] = transform(X_test[i])
    


In [12]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_test)):
    n, edge_type = X_test[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)



In [13]:
node_types = list(node_types)
edge_types = list(edge_types)

In [14]:
node_types

['DiagnosticIC',
 'timesincecasestart',
 'DiagnosticArtAstrup',
 'timesincemidnight',
 'Hypotensie',
 'SIRSCritTemperature',
 'month',
 'DiagnosticLacticAcid',
 'Activity',
 'Hypoxie',
 'DiagnosticBlood',
 'SIRSCriteria2OrMore',
 'DiagnosticUrinarySediment',
 'Leucocytes',
 'DiagnosticXthorax',
 'CRP',
 'SIRSCritLeucos',
 'Infusion',
 'DiagnosticUrinaryCulture',
 'DiagnosticECG',
 'timesincelastevent',
 'Diagnose',
 'DiagnosticSputum',
 'Age',
 'SIRSCritHeartRate',
 'hour',
 'event_nr',
 'InfectionSuspected',
 'weekday',
 'org:group',
 'SIRSCritTachypnea',
 'DisfuncOrg',
 'DiagnosticLiquor',
 'Oligurie',
 'LacticAcid',
 'open_cases',
 'time:timestamp',
 'DiagnosticOther']

In [15]:
edge_types

[('Activity', 'related_to', 'Infusion'),
 ('InfectionSuspected', 'related_to', 'InfectionSuspected'),
 ('DiagnosticXthorax', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'DisfuncOrg'),
 ('Activity', 'related_to', 'SIRSCriteria2OrMore'),
 ('Activity', 'related_to', 'SIRSCritTemperature'),
 ('event_nr', 'rev_related_to', 'Activity'),
 ('Diagnose', 'related_to', 'Diagnose'),
 ('DiagnosticXthorax', 'related_to', 'DiagnosticXthorax'),
 ('Activity', 'related_to', 'Leucocytes'),
 ('Activity', 'related_to', 'time:timestamp'),
 ('Activity', 'related_to', 'DiagnosticUrinaryCulture'),
 ('Activity', 'related_to', 'Hypotensie'),
 ('hour', 'related_to', 'hour'),
 ('Activity', 'related_to', 'DiagnosticSputum'),
 ('DisfuncOrg', 'related_to', 'DisfuncOrg'),
 ('org:group', 'rev_related_to', 'Activity'),
 ('Hypotensie', 'rev_related_to', 'Activity'),
 ('Diagnose', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'DiagnosticLacticAcid'),
 ('Activity', 'related_to', 'CRP'),
 ('month

## Hyperopt

In [16]:
from ax.service.managed_loop import optimize

In [17]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [18]:
from torch.nn import Module, ModuleList, Linear
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv, global_mean_pool

class HGNN(Module):
    def __init__(self, node_types, edge_types, parameters):
        super().__init__()
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]
        
        self.node_types = node_types
        self.edge_types = edge_types
        
        # Convolutional layers for heterogeneous graph
        self.convs = ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {relation: SAGEConv((-1, -1), hid, aggr=aggregation)
                 for relation in edge_types},
                aggr=aggregation
            )
            self.convs.append(conv)
        
        # Linear layer for graph-level prediction
        self.lin = Linear(len(node_types) * hid, 1)
    
    def forward(self, batch):
        x_dict = batch.x_dict
        edge_index_dict = batch.edge_index_dict
        
        # Apply convolutional layers
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}
        
        # Pool node features for each node type
        graph_features = []
        for node_type in self.node_types:
            x = x_dict[node_type]
            batch_idx = batch[node_type].batch  # Batch index for pooling
            pooled = global_mean_pool(x, batch_idx)
            graph_features.append(pooled)
        
        # Concatenate pooled features
        graph_features = torch.cat(graph_features, dim=-1)
        
        # Predict remaining time
        output = self.lin(graph_features).squeeze(-1)
        return output  # Shape: [batch_size]

In [19]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score

In [20]:
import torch.nn as nn

In [21]:
import time

In [22]:
from torch_geometric.data import DataLoader
from copy import deepcopy

def train_hgnn(config, node_types, edge_types, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = HGNN(node_types=node_types, edge_types=edge_types, parameters=config).to(device)
    loss_fn = nn.L1Loss()  # MAE for regression
    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])
    
    train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)
    
    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0
    
    for epoch in range(epochs):
        net.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            preds = net(batch)  # Graph-level predictions
            true = batch.y  # Graph-level targets
            loss = loss_fn(preds, true)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        avg_train_loss = sum(train_losses) / len(train_losses)
        
        net.eval()
        valid_losses = []
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(device)
                preds = net(batch)
                true = batch.y
                valid_losses.append(loss_fn(preds, true).item())
        
        avg_val_loss = sum(valid_losses) / len(valid_losses)
        
        print(f"Epoch {epoch+1}/{epochs}, Train MAE: {avg_train_loss:.4f}, Valid MAE: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_model = deepcopy(net)
            pat_count = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                print("Early stopping")
                break
    
    return best_model

In [23]:
def test_hgnn(net):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    all_preds = []
    all_targets = []
    net.eval()
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            preds = net(batch)
            all_preds.append(preds)
            all_targets.append(batch.y)
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae = nn.L1Loss()(all_preds, all_targets).item()
    print(f"Test MAE: {mae:.4f}")
    return {"remaining_time_mae": mae}

In [24]:
outputreal = ["remaining_time"]
print(outputreal)

['remaining_time']


In [25]:
def train_evaluate(config):
    trained_net = train_hgnn(config, node_types=node_types,edge_types=edge_types, epochs=50)
    return test_hgnn(trained_net)

In [26]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [27]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}

# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config,node_types=node_types,edge_types=edge_type, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)

Epoch 1/1, Train MAE: 670347.8750, Valid MAE: 713625.8750
Test MAE: 649003.3125
test_hgnn returned: {'remaining_time_mae': 649003.3125}


In [28]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=lambda config:train_evaluate({**config,"nodes_relations": edge_types}),
    objective_name='remaining_time_mae',
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = 15
)

print("Best parameters:", best_parameters)
means, covariances = values
print("Means", means)
print("Experiment:", experiment)

[INFO 06-19 19:38:53] ax.service.utils.instantiation: Choice parameter hid contains only one value, converting to a fixed parameter instead.
[INFO 06-19 19:38:53] ax.service.utils.instantiation: Choice parameter layers contains only one value, converting to a fixed parameter instead.
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWa

Epoch 1/50, Train MAE: 649527.3438, Valid MAE: 713627.1875
Epoch 2/50, Train MAE: 663657.5938, Valid MAE: 713626.8125
Epoch 3/50, Train MAE: 672171.8438, Valid MAE: 713626.5000
Epoch 4/50, Train MAE: 649544.0625, Valid MAE: 713626.1875
Epoch 5/50, Train MAE: 654446.3750, Valid MAE: 713625.8750
Epoch 6/50, Train MAE: 654881.6875, Valid MAE: 713625.5000
Epoch 7/50, Train MAE: 673858.1875, Valid MAE: 713625.1250
Epoch 8/50, Train MAE: 659671.2812, Valid MAE: 713624.7500
Epoch 9/50, Train MAE: 666171.7500, Valid MAE: 713624.3750
Epoch 10/50, Train MAE: 655997.7812, Valid MAE: 713623.9375
Epoch 11/50, Train MAE: 675508.6875, Valid MAE: 713623.4375
Epoch 12/50, Train MAE: 660783.3750, Valid MAE: 713623.0000
Epoch 13/50, Train MAE: 658108.1562, Valid MAE: 713622.3750
Epoch 14/50, Train MAE: 665521.3125, Valid MAE: 713621.8750
Epoch 15/50, Train MAE: 667973.2812, Valid MAE: 713621.2500
Epoch 16/50, Train MAE: 654138.5000, Valid MAE: 713620.5625
Epoch 17/50, Train MAE: 670232.8125, Valid MAE: 7

[INFO 06-19 19:41:00] ax.service.managed_loop: Running optimization trial 2...


Test MAE: 648929.6250
Epoch 1/50, Train MAE: 638850.0625, Valid MAE: 598845.8125
Epoch 2/50, Train MAE: 484338.8359, Valid MAE: 488593.2500
Epoch 3/50, Train MAE: 413217.4297, Valid MAE: 412528.4688
Epoch 4/50, Train MAE: 383293.1953, Valid MAE: 437735.6875
Epoch 5/50, Train MAE: 403725.0156, Valid MAE: 396761.2500
Epoch 6/50, Train MAE: 353709.2578, Valid MAE: 392624.3438
Epoch 7/50, Train MAE: 338673.8438, Valid MAE: 404900.3125
Epoch 8/50, Train MAE: 334018.6953, Valid MAE: 390219.5938
Epoch 9/50, Train MAE: 356459.3438, Valid MAE: 388733.4375
Epoch 10/50, Train MAE: 338285.8672, Valid MAE: 386350.6250
Epoch 11/50, Train MAE: 346513.7109, Valid MAE: 386494.6250
Epoch 12/50, Train MAE: 377061.7266, Valid MAE: 385192.6250
Epoch 13/50, Train MAE: 310647.7539, Valid MAE: 383149.2188
Epoch 14/50, Train MAE: 313928.8594, Valid MAE: 382651.5000
Epoch 15/50, Train MAE: 347682.9375, Valid MAE: 384432.8125
Epoch 16/50, Train MAE: 305416.2812, Valid MAE: 389632.8438
Epoch 17/50, Train MAE: 289

[INFO 06-19 19:41:54] ax.service.managed_loop: Running optimization trial 3...


...

Test MAE: 337982.8750
Epoch 1/50, Train MAE: 673955.5938, Valid MAE: 711859.6875
Epoch 2/50, Train MAE: 667997.0000, Valid MAE: 704404.2500
Epoch 3/50, Train MAE: 640990.5938, Valid MAE: 685544.3750
Epoch 4/50, Train MAE: 625908.4688, Valid MAE: 647858.8125
Epoch 5/50, Train MAE: 581245.5938, Valid MAE: 584410.8125
Epoch 6/50, Train MAE: 534103.2031, Valid MAE: 489731.1562
Epoch 7/50, Train MAE: 418369.5625, Valid MAE: 415210.7500
Epoch 8/50, Train MAE: 368535.2656, Valid MAE: 405060.0000
Epoch 9/50, Train MAE: 392001.9844, Valid MAE: 435828.6875
Epoch 10/50, Train MAE: 409818.5781, Valid MAE: 427108.8438
Epoch 11/50, Train MAE: 397751.0781, Valid MAE: 401680.7500
Epoch 12/50, Train MAE: 361105.5781, Valid MAE: 399937.3438
Epoch 13/50, Train MAE: 349895.3750, Valid MAE: 413620.5000
Epoch 14/50, Train MAE: 367951.0156, Valid MAE: 422709.7812
Epoch 15/50, Train MAE: 387730.3750, Valid MAE: 420984.5625
Epoch 16/50, Train MAE: 361776.3750, Valid MAE: 411082.2812
Epoch 17/50, Train MAE: 347

[INFO 06-19 19:59:16] ax.service.managed_loop: Running optimization trial 14...


Test MAE: 336756.3750
Epoch 1/50, Train MAE: 663310.0000, Valid MAE: 709293.6875
Epoch 2/50, Train MAE: 658984.4375, Valid MAE: 695991.3750
Epoch 3/50, Train MAE: 645775.0625, Valid MAE: 666880.0000
Epoch 4/50, Train MAE: 617159.7500, Valid MAE: 614975.2500
Epoch 5/50, Train MAE: 566916.2500, Valid MAE: 536383.5625
Epoch 6/50, Train MAE: 491249.6562, Valid MAE: 441456.8125
Epoch 7/50, Train MAE: 396941.7812, Valid MAE: 399874.2500
Epoch 8/50, Train MAE: 356881.6250, Valid MAE: 432776.1250
Epoch 9/50, Train MAE: 414103.8125, Valid MAE: 459022.6562
Epoch 10/50, Train MAE: 446855.6562, Valid MAE: 445577.7812
Epoch 11/50, Train MAE: 430421.3438, Valid MAE: 415491.9688
Epoch 12/50, Train MAE: 390953.3750, Valid MAE: 396341.8438
Epoch 13/50, Train MAE: 359826.2188, Valid MAE: 404676.3438
Epoch 14/50, Train MAE: 354618.1875, Valid MAE: 420248.8438
Epoch 15/50, Train MAE: 370724.2188, Valid MAE: 432081.0000
Epoch 16/50, Train MAE: 385928.9375, Valid MAE: 434730.2812
Epoch 17/50, Train MAE: 389

[INFO 06-19 20:00:04] ax.service.managed_loop: Running optimization trial 15...


Test MAE: 341731.7812
Epoch 1/50, Train MAE: 701098.2344, Valid MAE: 712053.3125
Epoch 2/50, Train MAE: 658408.2656, Valid MAE: 702343.1875
Epoch 3/50, Train MAE: 632622.8125, Valid MAE: 672635.5625
Epoch 4/50, Train MAE: 611287.8047, Valid MAE: 605839.3750
Epoch 5/50, Train MAE: 512686.5312, Valid MAE: 487162.7188
Epoch 6/50, Train MAE: 419710.8906, Valid MAE: 404042.4688
Epoch 7/50, Train MAE: 372942.2031, Valid MAE: 422084.5625
Epoch 8/50, Train MAE: 383114.4922, Valid MAE: 413065.9062
Epoch 9/50, Train MAE: 360323.2656, Valid MAE: 399318.4688
Epoch 10/50, Train MAE: 354842.8594, Valid MAE: 416948.0625
Epoch 11/50, Train MAE: 368147.9375, Valid MAE: 419690.2188
Epoch 12/50, Train MAE: 361607.8281, Valid MAE: 406207.2500
Epoch 13/50, Train MAE: 341326.8906, Valid MAE: 394898.9375
Epoch 14/50, Train MAE: 349211.4453, Valid MAE: 393910.0938
Epoch 15/50, Train MAE: 362669.6172, Valid MAE: 392720.4062
Epoch 16/50, Train MAE: 350131.6406, Valid MAE: 394157.5312
Epoch 17/50, Train MAE: 335

In [29]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)

[WARNING 06-19 20:01:50] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


In [30]:
results.sort_values(by="remaining_time_mae")

,trial_index,arm_name,trial_status,generation_method,remaining_time_mae,lr,batch_size,aggregation,hid,layers
7,7,7_0,COMPLETED,BO_MIXED,328314.28125,0.100000,128,max,128,2
1,1,1_0,COMPLETED,Sobol,330127.90625,0.042125,128,sum,128,2
8,8,8_0,COMPLETED,BO_MIXED,330205.00000,0.100000,128,mean,128,2
14,14,14_0,COMPLETED,BO_MIXED,332705.78125,0.031305,128,mean,128,2
12,12,12_0,COMPLETED,BO_MIXED,336756.37500,0.014951,256,sum,128,2
11,11,11_0,COMPLETED,BO_MIXED,337982.87500,0.027176,512,sum,128,2
5,5,5_0,COMPLETED,Sobol,338728.71875,0.013215,128,mean,128,2
13,13,13_0,COMPLETED,BO_MIXED,341731.78125,0.100000,512,max,128,2
6,6,6_0,COMPLETED,BO_MIXED,344461.90625,0.005030,128,sum,128,2
4,4,4_0,COMPLETED,Sobol,370438.81250,0.002844,512,sum,128,2


In [31]:
results = results.sort_values(by="remaining_time_mae")

In [32]:
results.to_csv(f"results/{dataset}.csv", sep=",")